# Module 5: Code Architecture & Implementation Walkthrough

**Estimated time: 45 minutes**

In this module, we read and analyze the actual Model Swarms source code. Keep the [repository](https://github.com/BunsenFeng/model_swarm) open alongside this notebook.

## 5.1 Repository Structure

```
model_swarm/
├── search.py              # Main algorithm: PSO search loop
├── evaluate.py            # Utility functions: model evaluation
├── merge.py               # Core operation: weighted LoRA merging
├── overall_metrics.py     # Post-search analysis and metrics
├── reward_modeling.py     # Reward model scoring (Objective 3)
├── search_nlgraph.sh      # Example: single-task search (NLGraph)
├── search_legal.sh        # Example: multi-task domain (Legal)
├── search_concise.sh      # Example: reward model (concise)
├── search_phd_application.sh  # Example: human interest
├── data/eval/             # Evaluation datasets (JSON format)
├── initial_experts/       # LoRA adapters + download script
├── search/                # Output directory (created at runtime)
├── swarm.yml              # Conda environment specification
└── LICENSE                # Apache 2.0
```

The codebase is compact — the entire algorithm fits in four Python files totaling ~1200 lines.

## 5.2 `merge.py` — The Atomic Operation

This is the smallest and most important file. Every operation in Model Swarms is expressed as a call to `lora_merge()`.

**Two code paths:**

| Path | When | How | Speed |
|------|------|-----|-------|
| Slow merge | `directly_load_safetensors=0` | Loads full PEFT models via `AutoModelForCausalLM` | Slow (loads 7B model) |
| Fast merge | `directly_load_safetensors=1` | Loads only safetensor files | Fast (loads ~70MB) |

The fast merge path (always used in practice):

```python
# Load raw tensors, compute weighted sum, save
lora_state_dict_list = []
for lora_name in lora_name_list:
    state_dict = load_file(
        os.path.join(lora_name, "adapter_model.safetensors"),
        device="cpu"
    )
    lora_state_dict_list.append(state_dict)

final_state_dict = {}
for i in range(len(lora_state_dict_list)):
    for key in lora_state_dict_list[i].keys():
        if i == 0:
            final_state_dict[key] = weights[i] * lora_state_dict_list[i][key]
        else:
            final_state_dict[key] += weights[i] * lora_state_dict_list[i][key]

save_file(final_state_dict, os.path.join(output_name, "adapter_model.safetensors"))
```

**Key observation**: Merging is done on CPU (`device="cpu"`). GPUs are reserved for model evaluation (inference).

## 5.3 `evaluate.py` — The Utility Functions

The main entry point handles all four evaluation objectives:

```python
def evaluate(model_path, eval_type, dataset, gpu_id,
             base_model="google/gemma-7b-it", ...):
```

**Evaluation types:**

| `eval_type` | Method | Output |
|-------------|--------|--------|
| `multiple_choice` | Parse model output for letter answer (A/B/C/D) | Accuracy score |
| `exact_match` | Check if gold answer appears in model output | Fraction correct |
| `external_api` | Generate text, score with Perspective API (toxicity) | 1 - toxicity score |
| `AbstainQA` | Two-pass: answer + self-check | Reliability metric |
| `multitask` | Run two evaluations, return harmonic mean | Harmonic mean |
| `rm_*` | Generate responses, score with reward model | Average RM score |
| `human` | Generate responses, score with Gemini-as-judge | Average 1-10 rating |

Key implementation details:
- `do_sample=False` — all generation is **deterministic** (greedy decoding), ensuring the utility function is deterministic for the same model
- Batch generation with explicit `torch.cuda.empty_cache()` between batches
- Cascading answer parser tries multiple strategies in decreasing reliability order

## 5.4 `search.py` — The Main Algorithm

This is the largest file (~400 lines). Let's walk through the key sections.

### Directory Structure Created at Runtime

```
search/<experiment_name>/
├── particle_0/
│   ├── now/           # current position (LoRA weights)
│   ├── personal_best/ # best position visited
│   └── velocity/      # current velocity (LoRA weights)
├── particle_1/
│   ├── now/
│   ├── personal_best/
│   └── velocity/
├── ...
├── global_best/       # best position across all particles
├── global_worst/      # worst position across all particles
└── utility_scratchpad.json  # all scores and history
```

Each directory contains an `adapter_model.safetensors` file — a set of LoRA weights.

### The Main Search Loop (Annotated)

```python
iter_count = 0
while iter_count < max_iteration:
    iter_count += 1

    # === PATIENCE CHECK (EARLY STOPPING) ===
    g_history = utility_scratchpad["g_history"]
    if len(g_history) > patience:
        recent = g_history[-patience:]
        if max(recent) == min(recent):  # no improvement
            break

    # === RESTART CHECK ===
    for i in range(len(particle_paths)):
        if restart_stray_particles:
            # Has this particle stagnated?
            restart_flag = (stagnation_time >= restart_patience * patience)

    # === VELOCITY + POSITION UPDATE (CPU-parallelized) ===
    update_args = [(i, gpu_id, ..., restart_flag) for i in range(N)]
    pool = Pool(processes=num_cpu_when_merging)
    pool.starmap(particle_update, update_args)

    # === EVALUATION (GPU-parallelized) ===
    eval_args = [(particle_path, eval_type, dataset, gpu_id, ...) for ...]
    pool = Pool(processes=len(gpus))
    results = pool.starmap(evaluate, eval_args)

    # === UPDATE TRACKING ===
    for i in range(N):
        if results[i] > personal_best[i]:
            update personal_best[i]
        if results[i] > global_best:
            update global_best
        if results[i] < global_worst:
            update global_worst

    # === DECAY STEP LENGTH ===
    step_length = max(step_length * step_length_factor, minimum_step_length)
```

### Parallelization Strategy

| Operation | Parallelized How | Why |
|-----------|-----------------|-----|
| Merge operations | `Pool(processes=num_cpu_when_merging)` | Merging is CPU-bound (tensor arithmetic) |
| Evaluations | `Pool(processes=len(gpus))` | Evaluation is GPU-bound (model inference) |

With 20 particles and 5 GPUs: particles 0-3 on GPU 0, 4-7 on GPU 1, etc.

## 5.5 Data Flow Diagram

```
┌─────────────────────────────────────────────────┐
│                    search.py                     │
│                                                  │
│  initial_experts/ ──────────────────────────┐    │
│       │                                     │    │
│       ▼                                     │    │
│  [Population Expansion] ←── merge.py        │    │
│       │                                     │    │
│       ▼                                     │    │
│  [Initial Evaluation] ←── evaluate.py       │    │
│       │                                     │    │
│       ▼                                     │    │
│  ┌─────────────────────┐                    │    │
│  │ Search Loop (×K)    │                    │    │
│  │                     │                    │    │
│  │ 1. Velocity Update  │ ←── merge.py (×6N)│    │
│  │ 2. Position Update  │ ←── merge.py (×N) │    │
│  │ 3. Evaluation       │ ←── evaluate.py   │    │
│  │ 4. Update trackers  │                    │    │
│  │ 5. Check patience   │                    │    │
│  │ 6. Decay step length│                    │    │
│  └─────────────────────┘                    │    │
│       │                                     │    │
│       ▼                                     │    │
│  [Final Evaluation] ←── overall_metrics.py  │    │
│       │                                     │    │
│       ▼                                     │    │
│  search/<name>/global_best/                  │    │
└─────────────────────────────────────────────────┘
```

## 5.6 Key Design Decisions and Trade-offs

### Decision 1: File-System State Management
All state (positions, velocities, personal bests) is stored as files on disk, not in memory.
- **Pro**: Survives crashes; easy to inspect/debug; memory-efficient
- **Con**: I/O overhead from constant read/write of safetensor files

### Decision 2: Multiprocessing for Parallelism
Python's `multiprocessing.Pool` rather than distributed computing.
- **Pro**: Simple; works on a single machine
- **Con**: Limited to one machine's GPUs

### Decision 3: Random Hyperparameter Search
Shell scripts use random search rather than grid search or Bayesian optimization.
- Each run is independent and logged to W&B
- Bad hyperparameter combinations terminate quickly (patience kicks in)

## Exercise 5.1: Code Reading Questions

Open [search.py](https://github.com/BunsenFeng/model_swarm/blob/main/search.py) and answer:

1. What happens if `populate_initial_experts=0`? How many particles will the search use?
2. Find the line where `wandb.log()` is called. What metrics are logged per iteration?
3. Why is `torch.multiprocessing.set_start_method('spawn')` called instead of `fork`?

*Your answers:*

1. 
2. 
3. 

## Exercise 5.2: Evaluation Pipeline Analysis

Study [evaluate.py](https://github.com/BunsenFeng/model_swarm/blob/main/evaluate.py) and answer:

1. Why does multiple_choice evaluation use `max_new_tokens=10` instead of `max_new_tokens=1`?
2. In the GSM8k exact_match, why does the code take the last 5 words before checking for the answer?
3. The AbstainQA evaluation involves two inference passes. What does each pass do?

*Your answers:*

1. 
2. 
3. 

## Exercise 5.3: Architectural Improvement Proposals

Identify one limitation of the current implementation and propose a concrete improvement. Consider:
- Memory efficiency (could velocity be stored implicitly?)
- Parallelization (distributed across multiple machines?)
- Convergence (adaptive hyperparameter scheduling?)
- Code quality (which parts need refactoring?)

Write a short (1-2 paragraph) design proposal.

*Your proposal:*



---

**Next: [Module 6 — Hands-On Lab: Running Experiments](module_06_hands_on_lab.ipynb)**